# 96 — Campaign B end-to-end（任意・スタンドアロン統合ノート）

**スタンドアロンの二次オプション。** `run_end_to_end` による統一 end-to-end。  
推奨運用は **89∥97**（producer/consumer）。96 は **89∥97 の代替**であり、  
**97 と同時フル起動しない**（同一 GPU lane lease）。単独で使うなら OK。

- M3/downstream 優先
- 任意: `list_gpu_m3_queue` 長 < `selected_backlog_target` のときだけ screen+advance
- progress は完了のみ（`m3_checkpoint` 単独は数えない）
- `VALIDATED_RG_DISABLE_SESSION_WALLCLOCK=1`
- production gate 81 禁止 / `NOT_CERTIFIED` / `SCREENING_ONLY`
- GPU lane lease: `campaign_b/_locks/gpu_lane.json` — **97 と同時実行禁止**
- **M3 auto-strip（既定 ON）:** セッション開始時に COMPLETE+downstream をフル安全
  スキャンして strip。各ラウンド後も増分 strip。reports は残す。
- **Keep-latest（既定 ON）:** セッション開始時に **全** `runs/M3-*` へフル keep-latest
  （このバッチで触れない古い M3 の ckpt 積み上げも回収）。加えて各 M3 セッションの
  resume 前・後でも per-run trim。
- knobs: `AUTO_STRIP_M3_CHECKPOINTS=True`, `AUTO_KEEP_LATEST_M3_CHECKPOINT=True`,
  `PERSIST_M3_CAP_GIB=32.0`。詳細: `docs/campaign_b_m3_storage_reclaim.md`

設計: `docs/campaign_b_end_to_end_design.md`（推奨 89∥97 は `docs/campaign_b_parallel_split_design.md`）


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'end_to_end.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/end_to_end.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ['VALIDATED_RG_DISABLE_SESSION_WALLCLOCK'] = '1'
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 96 requires CUDA for M3/M4 stages.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.end_to_end import load_end_to_end_config

validate_persistent_root()
CFG_PATH = PROJECT_ROOT / 'configs' / 'campaign_b_end_to_end.yaml'
CFG = load_end_to_end_config(
    CFG_PATH,
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
)
# Optional knobs:
# CFG.selected_backlog_target = 8
# CFG.screening_chunk_size = 32
# CFG.max_rounds = 100
# After M4+ consumes M3: strip M3 checkpoints (fail-closed). Default ON.
# Session start always runs a full safe strip scan (backlog reclaim).
# See docs/campaign_b_m3_storage_reclaim.md
AUTO_STRIP_M3_CHECKPOINTS = True
# Session start: full keep-latest over ALL runs/M3-* (not only this batch).
# Also per M3 session before/after run_one_gpu_m3 (stops mid-run re-accumulation).
AUTO_KEEP_LATEST_M3_CHECKPOINT = True
# Cap total runs/M3-* size (GiB). None disables. Paperspace 15GB included + billing.
PERSIST_M3_CAP_GIB = 32.0
CFG.auto_strip_m3_checkpoints = AUTO_STRIP_M3_CHECKPOINTS
CFG.auto_keep_latest_m3_checkpoint = AUTO_KEEP_LATEST_M3_CHECKPOINT
CFG.persist_m3_cap_gib = PERSIST_M3_CAP_GIB
print(json.dumps({
    'selected_backlog_target': CFG.selected_backlog_target,
    'screening_chunk_size': CFG.screening_chunk_size,
    'max_rounds': CFG.max_rounds,
    'm3_backend': CFG.m3_backend,
    'disable_session_wallclock': CFG.disable_session_wallclock,
    'AUTO_STRIP_M3_CHECKPOINTS': AUTO_STRIP_M3_CHECKPOINTS,
    'AUTO_KEEP_LATEST_M3_CHECKPOINT': AUTO_KEEP_LATEST_M3_CHECKPOINT,
    'PERSIST_M3_CAP_GIB': PERSIST_M3_CAP_GIB,
}, indent=2))


In [ ]:
from src.campaign_b.end_to_end import run_end_to_end

SESSION = run_end_to_end(CFG)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'rounds_run': SESSION.get('rounds_run'),
    'auto_strip_m3_checkpoints': SESSION.get('auto_strip_m3_checkpoints'),
    'auto_keep_latest_m3_checkpoint': SESSION.get('auto_keep_latest_m3_checkpoint'),
    'persist_m3_cap_gib': SESSION.get('persist_m3_cap_gib'),
    'm3_reclaim': SESSION.get('m3_reclaim'),
    'totals': SESSION.get('totals'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_end_to_end' / 'LATEST_END_TO_END_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))


In [ ]:
# Round detail (progress / backlog gate)
print(json.dumps([
    {
        'round': r.get('round'),
        'progress': r.get('progress'),
        'm3_queue_len': r.get('m3_queue_len'),
        'backlog_gate_open': r.get('backlog_gate_open'),
        'stages': {k: {sk: sv for sk, sv in (v or {}).items() if sk != 'errors'}
                   for k, v in (r.get('stages') or {}).items()},
    }
    for r in (SESSION.get('rounds') or [])
], indent=2, ensure_ascii=False, default=str))
